# Modbus — master, slave and simulator

**What it is.** Modbus is the request/response lingua franca of PLCs, meters, drives and sensors. A *master* (client) asks; a *slave* (server) answers from four tables: coils, discrete inputs, input registers, holding registers.

## Setup
> Working from a clone? Run `dotnet pack -c Release -o artifacts/packages` at the repo root and add
> `#i "nuget: <repo>/artifacts/packages"` before the `#r` lines.

In [ ]:
#r "nuget: IoTCom.Net, 0.6.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.6.0-preview.1"

## Server: a virtual PLC
`ModbusSimulator.CreateVirtualPlc` animates temperature, motor speed, power and a production counter.

In [ ]:
using IoTCom.Net;
using IoTCom.Net.Protocols.Modbus;
using IoTCom.Net.Transports;

var link = new InMemoryTransportListener();
var store = new ModbusDataStore();
var plcServer = ModbusServer.Create(o => o.ListenInMemory(link).WithStore(store));
var simulator = ModbusSimulator.CreateVirtualPlc(store);
await plcServer.StartAsync();
for (var i = 0; i < 20; i++) simulator.Tick(0.25); // deterministic time
Console.WriteLine($"temperature register = {store.InputRegisters[0]}");

## Client: read and convert
Registers are 16-bit; 32-bit floats span two registers. `ModbusConvert` handles the four word orders found in the wild.

In [ ]:
var plc = ModbusClient.Create(o => o.UseInMemory(link).WithUnitId(1));
var ir = await plc.ReadInputRegistersAsync(0, 8);
Console.WriteLine($"temperature {ir[0] / 10.0} °C, rpm {ir[3]}, power {ModbusConvert.ToSingle(ir.AsSpan(4, 2)):0.00} kW");

## Write — and the read-only safety mode
Writes change real equipment. `.AsReadOnly()` blocks them before anything reaches the wire.

In [ ]:
await plc.WriteSingleCoilAsync(0, false); // stop the motor
for (var i = 0; i < 20; i++) simulator.Tick(0.25);
Console.WriteLine($"rpm after stop: {(await plc.ReadInputRegistersAsync(3, 1))[0]}");

var monitor = ModbusClient.Create(o => o.UseInMemory(link).AsReadOnly());
try { await monitor.WriteSingleRegisterAsync(0, 300); } catch (ReadOnlyModeException e) { Console.WriteLine(e.Message); }

## The Rust engine
`NativeModbusClient` runs the same protocol through the sans-I/O Rust state machine. Same API, identical wire bytes.

In [ ]:
using IoTCom.Net.Native.Modbus;
if (NativeModbusMaster.IsSupported)
{
    await using var rust = NativeModbusClient.Create(o => o.UseInMemory(link));
    Console.WriteLine($"via Rust: {string.Join(", ", await rust.ReadHoldingRegistersAsync(0, 3))}");
}
else Console.WriteLine("Native library not available for this platform — the managed client covers everything.");

## Troubleshooting
- **Timeout**: wrong unit id, wrong port, or RTU parity/baud mismatch (Modbus RTU default is 8E1).
- **IllegalDataAddress**: address + count beyond the device map — check 0-based vs 1-based (40001 = HR0).
- **Garbage on RTU**: two masters on one bus, or missing termination resistors.

## Exercise
Add `simulator.AddSignal(ModbusTable.HoldingRegisters, 50, t => Math.Sin(t) * 100 + 100)` and plot HR50 over time.

*IoTCom.Net — built by Gravicode Studios, led by Kang Fadhil.*